In [13]:
import sys
from pathlib import Path

for candidate in (Path.cwd(), *Path.cwd().parents):
    if (candidate / "src" / "tdlib").is_dir():
        sys.path.insert(0, str(candidate))
        break
else:
    raise FileNotFoundError("Could not find the project root")

## Fetching Telegram Groups

In [24]:
import importlib
import src.tdlib.client as tdlib_client
from config.settings import settings

importlib.reload(tdlib_client)
TdlibClient = tdlib_client.TdlibClient

In [23]:
client = TdlibClient(
    api_id=settings.telegram_api_id,
    api_hash=settings.telegram_api_hash,
    phone_number=settings.telegram_phone_number,
    database_directory=settings.tdlib_database_dir,
)

groups = []

await client.start_async()
try:
    group_ids = settings.telegram_group_ids or await client.joined_group_ids_async()
    for group_id in group_ids:
        group_title = await client.get_group_title_async(group_id)
        groups.append({
            "group_id": group_id,
            "group_title": group_title
        })
finally:
    await client.close_async()

/Users/yusufaxmad/Documents/ai/Job-Posting-Analysis/.venv/lib/python3.14/site-packages/telethon/client/auth.py:155: UserWarning: the session already had an authorized user so it did not login to the user account using the provided phone; if you were expecting a different user, check whether you are accidentally reusing an existing session
  warnings.warn(


In [38]:
import pandas as pd

gr_df = pd.DataFrame(groups)
msg_df = pd.read_csv("../../data/telegram_messages.csv")

value_counts = msg_df["group_title"].value_counts()
last_msg_dates = msg_df.groupby(by="group_title").date.max()

message_counts = value_counts.rename(
    index=lambda x: x.removeprefix("group_").removesuffix(".jsonl")
)

last_msg_dates = last_msg_dates.rename(
    index=lambda x: x.removeprefix("group_").removesuffix(".jsonl")
)

gr_df["message_count"] = (
    gr_df["group_id"].astype(str)
    .map(message_counts)
    .fillna(0)
    .astype(int)
)

gr_df["last_msg_date"] = (
    gr_df["group_id"].astype(str)
    .map(last_msg_dates)
)

gr_df.to_csv("../../data/telegram_groups.csv", index=False)


gr_df

,group_id,group_title,message_count,last_msg_date
0,-1001544370991,KOREA HALAL WORKS ⚒️,1401,2026-09-24 18:37:31+00:00
1,-1002553359624,Ish e'lonlari,0,NaN
2,-1001548974049,Koreadagi uzbeklar E9.Bir biringizga nafingiz ...,51792,2026-09-24 14:01:04+00:00
3,-1001857660612,Mokpo ish e'lon,46388,2026-09-24 18:16:31+00:00
4,-1001141152967,Gachon Uzbekistan Students,22760,2026-09-24 13:18:56+00:00
5,-1002525696680,Seongnam ish elon,3306,2026-09-24 13:31:01+00:00
6,-1002166549165,Seul va Kyongido faqat ish e'lonlari,1882,2026-09-24 16:09:59+00:00
7,-1002643935758,서울 일 아르바이트 Seuldagi kunlik ishlar,607,2026-09-22 06:15:35+00:00
8,-1001742386027,Tegu Kyongsan Zavod 📦 Arbayt gruppa,34994,2026-09-24 18:05:41+00:00


## Labeling

In [60]:
jobs = msg_df["text"].str.contains(
    r"ish bor|иш бор|kiwi kk|kishi kerak",
    case=False,
    na=False,
    regex=True,
)

In [45]:
apt = msg_df["text"].str.contains(
    r"uy bor|уй бор",
    case=False,
    na=False,
    regex=True
)

In [53]:
partner = msg_df["text"].str.contains(
    r"sherik kerak",
    case=False,
    na=False,
    regex=True
)

In [55]:
msg_df[jobs].iloc[100].text

'Ассалому алейкум ва рахматуллах ва баракатух ж1 виза ф 2.4.5.6 иш бор шахар Коджедо иш тури Чукчан палпанга старт 140.вон колган малумотлар шу номер 010.3944.8789'

In [61]:
sum(jobs)

8743

In [68]:
msg_df[jobs]["text"].str.len().min()

np.int64(6)

In [79]:
jobs_df = msg_df.loc[jobs]

mask = jobs_df["text"].notna() & (jobs_df["text"].str.len() >= 100)

result = jobs_df.loc[mask]

shortest = result.loc[result["text"].str.len().idxmin(), "text"]

print(shortest)

Xayirli kech E.9 otkazdelar Yangsangda... ordak, tovuq gosht zavodida ish bor qolgan malumotlar lich


In [80]:
jobs_zavod = msg_df["text"].str.contains(
    r"zavod",
    case=False,
    na=False,
    regex=True,
)

In [81]:
jobs_zavod.sum()

np.int64(3527)